# Category Review Notebook

Use this notebook to review and analyze categorized transactions.

**Purpose:**
- See category distribution and totals
- Identify transactions still needing review
- Inspect deductible amounts by category
- Check merchant memory coverage
- Spot-check rule assignments

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from staging.database import get_connection
con = get_connection()
print('Connected.')

In [ ]:
# --- Category summary ---
summary = con.execute("""
    SELECT
        ct.category,
        COUNT(*) as count,
        ROUND(SUM(nt.original_amount), 2) as total_amount,
        ROUND(SUM(ct.deductible_amount), 2) as total_deductible,
        ROUND(AVG(ct.confidence), 3) as avg_confidence
    FROM categorized_transactions ct
    JOIN normalized_transactions nt ON ct.transaction_id = nt.transaction_id
    WHERE nt.original_amount > 0
    GROUP BY ct.category
    ORDER BY total_deductible DESC
""").df()
summary

In [ ]:
# --- Transactions needing review ---
review_df = con.execute("""
    SELECT
        nt.transaction_date,
        nt.institution,
        nt.merchant_raw,
        nt.merchant_normalized,
        nt.original_amount,
        ct.category,
        ct.confidence,
        ct.rule_applied
    FROM categorized_transactions ct
    JOIN normalized_transactions nt ON ct.transaction_id = nt.transaction_id
    WHERE ct.review_required = TRUE
    ORDER BY nt.transaction_date
""").df()
print(f'{len(review_df)} transactions need review')
review_df.head(20)

In [ ]:
# --- Deductible amount summary ---
totals = con.execute("""
    SELECT
        ct.deductible_status,
        COUNT(*) as count,
        ROUND(SUM(nt.original_amount), 2) as total_charged,
        ROUND(SUM(ct.deductible_amount), 2) as total_deductible
    FROM categorized_transactions ct
    JOIN normalized_transactions nt ON ct.transaction_id = nt.transaction_id
    GROUP BY ct.deductible_status
    ORDER BY total_deductible DESC
""").df()
totals

In [ ]:
# --- Top merchants by deductible amount ---
con.execute("""
    SELECT
        nt.merchant_normalized,
        ct.category,
        COUNT(*) as transactions,
        ROUND(SUM(ct.deductible_amount), 2) as total_deductible
    FROM categorized_transactions ct
    JOIN normalized_transactions nt ON ct.transaction_id = nt.transaction_id
    WHERE ct.deductible_status IN ('full', 'partial')
    GROUP BY nt.merchant_normalized, ct.category
    ORDER BY total_deductible DESC
    LIMIT 25
""").df()

In [ ]:
# --- Rule coverage analysis ---
con.execute("""
    SELECT
        CASE
            WHEN rule_applied LIKE 'keyword:%' THEN 'keyword rule'
            WHEN rule_applied LIKE 'memory:%' THEN 'merchant memory'
            WHEN rule_applied = '' OR rule_applied IS NULL THEN 'no rule (fallback)'
            ELSE rule_applied
        END as rule_source,
        COUNT(*) as count,
        ROUND(AVG(confidence), 3) as avg_confidence
    FROM categorized_transactions
    GROUP BY rule_source
    ORDER BY count DESC
""").df()

In [ ]:
# --- Monthly deductible expenses ---
con.execute("""
    SELECT
        strftime(nt.transaction_date, '%Y-%m') as month,
        ct.category,
        COUNT(*) as count,
        ROUND(SUM(ct.deductible_amount), 2) as deductible
    FROM categorized_transactions ct
    JOIN normalized_transactions nt ON ct.transaction_id = nt.transaction_id
    WHERE ct.deductible_status IN ('full', 'partial')
      AND nt.transaction_date IS NOT NULL
    GROUP BY month, ct.category
    ORDER BY month, deductible DESC
""").df()